In [1]:
import numpy as np 
import pandas as pd 
import seaborn as sns 
import matplotlib.pyplot as plt
import sklearn

In [2]:
df  = pd.read_csv("Crop Recommendation dataset(npk dataset 1).csv")

In [3]:
df.head()

,N,P,K,temperature,humidity,ph,rainfall,label
0,90,42,43,20.879744,82.002744,6.502985,202.935536,rice
1,85,58,41,21.770462,80.319644,7.038096,226.655537,rice
2,60,55,44,23.004459,82.320763,7.840207,263.964248,rice
3,74,35,40,26.491096,80.158363,6.980401,242.864034,rice
4,78,42,42,20.130175,81.604873,7.628473,262.717340,rice


In [4]:
X = df.drop('label', axis = 1)
y = df.label

In [5]:
print(X)
print(y)

        N   P   K  temperature   humidity        ph    rainfall
0      90  42  43    20.879744  82.002744  6.502985  202.935536
1      85  58  41    21.770462  80.319644  7.038096  226.655537
2      60  55  44    23.004459  82.320763  7.840207  263.964248
3      74  35  40    26.491096  80.158363  6.980401  242.864034
4      78  42  42    20.130175  81.604873  7.628473  262.717340
...   ...  ..  ..          ...        ...       ...         ...
2195  107  34  32    26.774637  66.413269  6.780064  177.774507
2196   99  15  27    27.417112  56.636362  6.086922  127.924610
2197  118  33  30    24.131797  67.225123  6.362608  173.322839
2198  117  32  34    26.272418  52.127394  6.758793  127.175293
2199  104  18  30    23.603016  60.396475  6.779833  140.937041

[2200 rows x 7 columns]
0         rice
1         rice
2         rice
3         rice
4         rice
         ...  
2195    coffee
2196    coffee
2197    coffee
2198    coffee
2199    coffee
Name: label, Length: 2200, dtype: str


In [6]:
# 1. Define the logic function
def assign_season(row):
    # Kharif: High Rain + High Humidity
    if row['rainfall'] > 100 and row['humidity'] > 70:
        return 'Kharif'
    # Rabi: Lower Temp + Low Rain
    elif row['temperature'] < 23 and row['rainfall'] < 50:
        return 'Rabi'
    # Summer/Zaid: Everything else (usually hotter/drier transition)
    else:
        return 'Summer'

# 2. Apply it to your X dataframe
X['season'] = X.apply(assign_season, axis=1)

# 3. Quick check to see the result
print("New X columns:", X.columns.tolist())
print("\nSeason counts:\n", X['season'].value_counts())
print("\nFirst 5 rows with Season:")
print(X.head())

New X columns: ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall', 'season']

Season counts:
 season
Summer    1444
Kharif     735
Rabi        21
Name: count, dtype: int64

First 5 rows with Season:
    N   P   K  temperature   humidity        ph    rainfall  season
0  90  42  43    20.879744  82.002744  6.502985  202.935536  Kharif
1  85  58  41    21.770462  80.319644  7.038096  226.655537  Kharif
2  60  55  44    23.004459  82.320763  7.840207  263.964248  Kharif
3  74  35  40    26.491096  80.158363  6.980401  242.864034  Kharif
4  78  42  42    20.130175  81.604873  7.628473  262.717340  Kharif


In [7]:
X

,N,P,K,temperature,humidity,ph,rainfall,season
0,90,42,43,20.879744,82.002744,6.502985,202.935536,Kharif
1,85,58,41,21.770462,80.319644,7.038096,226.655537,Kharif
2,60,55,44,23.004459,82.320763,7.840207,263.964248,Kharif
3,74,35,40,26.491096,80.158363,6.980401,242.864034,Kharif
4,78,42,42,20.130175,81.604873,7.628473,262.717340,Kharif
...,...,...,...,...,...,...,...,...
2195,107,34,32,26.774637,66.413269,6.780064,177.774507,Summer
2196,99,15,27,27.417112,56.636362,6.086922,127.924610,Summer
2197,118,33,30,24.131797,67.225123,6.362608,173.322839,Summer
2198,117,32,34,26.272418,52.127394,6.758793,127.175293,Summer


In [8]:
X.drop(["temperature", 'humidity', 'rainfall'], inplace = True, axis = 1)

In [9]:
X

,N,P,K,ph,season
0,90,42,43,6.502985,Kharif
1,85,58,41,7.038096,Kharif
2,60,55,44,7.840207,Kharif
3,74,35,40,6.980401,Kharif
4,78,42,42,7.628473,Kharif
...,...,...,...,...,...
2195,107,34,32,6.780064,Summer
2196,99,15,27,6.086922,Summer
2197,118,33,30,6.362608,Summer
2198,117,32,34,6.758793,Summer


In [10]:
from sklearn.preprocessing import StandardScaler

# 1. Clean Column Names (Removes hidden spaces and makes everything lowercase)
X.columns = X.columns.str.strip().str.lower()

# 2. Check the Actual Season Labels
# (Ensures the column name matches our lowercase cleaning)
season_col = 'season' 

# 3. One-Hot Encoding
X_encoded = pd.get_dummies(X, columns=[season_col], prefix='season')

# 4. Define the numeric columns to scale (matching the new lowercase names)
num_cols = ['n', 'p', 'k', 'temperature', 'humidity', 'ph', 'rainfall']

# 5. Safety check: Only scale columns that actually exist
existing_cols = [col for col in num_cols if col in X_encoded.columns]

# 6. Apply scaling
scaler = StandardScaler()
X_encoded[existing_cols] = scaler.fit_transform(X_encoded[existing_cols])

# FINAL CHECK
print("Tournament Ready! Final Columns:")
print(X_encoded.columns.tolist())
X_encoded.head(2)

Tournament Ready! Final Columns:
['n', 'p', 'k', 'ph', 'season_Kharif', 'season_Rabi', 'season_Summer']


,n,p,k,ph,season_Kharif,season_Rabi,season_Summer
0,1.068797,-0.344551,-0.101688,0.043302,True,False,False
1,0.933329,0.140616,-0.141185,0.734873,True,False,False


In [11]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder

# 1. Encode the Target (y)
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# 2. The 70/15/15 Split
# Split 1: 70% Train, 30% for the rest
X_train, X_remainder, y_train, y_remainder = train_test_split(
    X_encoded, y_encoded, test_size=0.3, random_state=42
)

# Split 2: Divide the 30% into two equal 15% halves
X_val, X_test, y_val, y_test = train_test_split(
    X_remainder, y_remainder, test_size=0.5, random_state=42
)

# 3. Train the "Baseline" Fighter
# The model now handles multi-class logic automatically!
model_lr = LogisticRegression(max_iter=2000)
model_lr.fit(X_train, y_train)

# 4. Evaluate on Validation Set
y_val_pred = model_lr.predict(X_val)
val_accuracy = accuracy_score(y_val, y_val_pred)

print(f"Tournament Standings - Logistic Regression")
print(f"Validation Accuracy: {val_accuracy * 100:.2f}%")
print("-" * 30)
print(classification_report(y_val, y_val_pred, target_names=le.classes_))

Tournament Standings - Logistic Regression
Validation Accuracy: 73.03%
------------------------------
              precision    recall  f1-score   support

       apple       1.00      1.00      1.00        22
      banana       1.00      1.00      1.00        15
   blackgram       0.62      0.62      0.62        16
    chickpea       0.62      1.00      0.76        13
     coconut       0.75      0.92      0.83        13
      coffee       1.00      0.73      0.85        15
      cotton       0.81      0.87      0.84        15
      grapes       1.00      1.00      1.00        11
        jute       0.65      0.65      0.65        17
 kidneybeans       0.62      0.71      0.67        14
      lentil       0.33      0.55      0.41        11
       maize       0.77      1.00      0.87        10
       mango       0.67      0.91      0.77        11
   mothbeans       0.71      0.25      0.37        20
    mungbean       0.68      0.94      0.79        16
   muskmelon       0.53      0.69

In [12]:
# 1. Create a fresh X from the original dataframe
# (Assuming your original loaded data is in 'df')
X_baseline = df[['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']].copy()

# 2. Preprocessing (Cleaning & Scaling)
X_baseline.columns = X_baseline.columns.str.strip().str.lower()
scaler_ns = StandardScaler()
X_scaled_ns = scaler_ns.fit_transform(X_baseline)

# 3. Encode Target (y) if not already done
le = LabelEncoder()
y_encoded = le.fit_transform(df['label'])

# 4. The 70/15/15 Split (Using the exact same random_state=42)
X_train_ns, X_rem_ns, y_train_ns, y_rem_ns = train_test_split(
    X_scaled_ns, y_encoded, test_size=0.3, random_state=42
)
X_val_ns, X_test_ns, y_val_ns, y_test_ns = train_test_split(
    X_rem_ns, y_rem_ns, test_size=0.5, random_state=42
)

# 5. Train & Evaluate
model_ns = LogisticRegression(max_iter=2000)
model_ns.fit(X_train_ns, y_train_ns)

# 6. Final Score
val_acc_ns = model_ns.score(X_val_ns, y_val_ns)

print("--- TOURNAMENT RESULTS ---")
print(f"Model 1 (With Seasons): 73.03%") # Your previous result
print(f"Model 2 (No Seasons):   {val_acc_ns * 100:.2f}%")
print("-" * 26)

if (val_acc_ns * 100) > 73.03:
    print("Winner: Baseline (No Seasons)")
    print("Insight: The raw numbers were more precise than the 'Season' labels.")
else:
    print("Winner: Model with Seasons")
    print("Insight: Adding categorical context helped the model simplify the decision.")

--- TOURNAMENT RESULTS ---
Model 1 (With Seasons): 73.03%
Model 2 (No Seasons):   96.67%
--------------------------
Winner: Baseline (No Seasons)
Insight: The raw numbers were more precise than the 'Season' labels.
